**### Preprocessing Michael Flanderka ###**

Für das Preprocessing habe ich als Basis den Code aus dem Paper genommen, diesen aber um

In [ ]:
import numpy as np
import pandas as pd
import pickle
import gzip

class Preprocessor:
    def __init__(self, test_fold=0, val_fold=1):
        self.test_fold = test_fold
        self.val_fold = val_fold
        self.means1 = None
        self.std1 = None
        self.means2 = None
        self.std2 = None
        self.feat_filt = None

Wir initalisieren die Klasse Preprocessor. Dies ist nötig, da wir die normalisation mit verschiedenen Paramtern aufrufen können.

Die Anwendungsfälle sind wie folgt unterteilt:
norm, tanh und tanh_norm welches verschiedene Normalisierungsstrategien sind. Näheres dazu im Paper.

Die Daten werden in folgender Art und Weise genutzt:

Fold        | Verwendung                    | Variable im Code     
------------|-------------------------------|------------------------
Fold 0      | Testset (nach dem Training)   | X_test, y_test         
Fold 1      | Validierungsset für Tuning    | X_val, y_val           
Folds 2–4   | Trainingsdaten                | X_tr, y_tr             
Folds 1–4   | Trainingsdaten nach Tuning    | X_train, y_train       

2.2 in the paper states:

For data normalization we employed three different types of input normalization: (i) standardizing all inputs to zero mean and unit variance, (ii) standarizing and applying hyperbolic tangent and (iii) standardizing, hyperbolic tangent and standardizing again.

This corresponds to the following code

In [ ]:
    @staticmethod
    def normalize(X, means1=None, std1=None, means2=None, std2=None, feat_filt=None, norm='tanh_norm'):
        if std1 is None:                                                      #If std1 not given: calculate
            std1 = np.nanstd(X, axis=0)
        if feat_filt is None:                                                 #If standard deviation is 0 (non informative) throw data away
            feat_filt = std1!=0
        X = X[:,feat_filt]
        X = np.ascontiguousarray(X)                                           #Data array needs to be continuous
        if means1 is None:                                                    #Calc mean and standardize
            means1 = np.mean(X, axis=0)
        X = (X-means1)/std1[feat_filt]
        if norm == 'norm':                                                    #Now we start the actual normalization (corresponds to the end of 2.2 in the paper):
            return(X, means1, std1, feat_filt)
        elif norm == 'tanh':
            return(np.tanh(X), means1, std1, feat_filt)
        elif norm == 'tanh_norm':
            X = np.tanh(X)
            if means2 is None:
                means2 = np.mean(X, axis=0)
            if std2 is None:
                std2 = np.std(X, axis=0)
            X = (X-means2)/std2
            X[:,std2==0]=0

            # print("Original shape:", X.shape)
            # print("Features kept:", np.sum(feat_filt))

            return(X, means1, std1, means2, std2, feat_filt)

**Parameter:**

X: Eingabematrix (z. B. Merkmale von Proben)

means1, std1: Mittelwert und Standardabweichung der Originaldaten (für erste Normalisierung)

means2, std2: Mittelwert und Standardabweichung nach tanh (für zweite Normalisierung bei tanh_norm)

feat_filt: Filtermaske, die Features mit Standardabweichung = 0 ausschließt

norm: gewählte Normalisierungsstrategie – 'norm', 'tanh', oder 'tanh_norm'

In [ ]:
    @staticmethod
    def load_features(features_path='X.p.gz'):
        #contains the data in both feature ordering ways (drug A - drug B - cell line and drug B - drug A - cell line)
        #in the first half of the data the features are ordered (drug A - drug B - cell line)
        #in the second half of the data the features are ordered (drug B - drug A - cell line)
        with gzip.open(features_path, 'rb') as file:
            X = pickle.load(file)
        return X


    @staticmethod
    def load_labels(labels_path='labels.csv'):
        #contains synergy values and fold split (numbers 0-4)
        labels = pd.read_csv(labels_path, index_col=0)
        #labels are duplicated for the two different ways of ordering in the data
        labels = pd.concat([labels, labels])
        #In the end X contains both ordering ways, both labeled the same. This therefor prevents AB and BA from being in different folds and thereby prevents data leakage.
        return labels

Bevor wir mit dem eigentlichen Preprocessing beginnen merken wir uns die Labels und laden natürlich den original Datensatz.

In [ ]:
 def create_splits(self, norm_method = 'tanh_norm', features_path='X.p.gz', labels_path='labels.csv'):
        """Create train/val/test splits with normalization."""
        labels = self.load_labels(labels_path)
        X = self.load_features(features_path)

In [ ]:
        # Feature-Origin creations according to paper
        feature_origin = (
            ["ECFP_6"] * 1309 +
            ["phys-chem"] * 802 +
            ["toxicophore"] * 2276 +
            ["ECFP_6"] * 1309 +
            ["phys-chem"] * 802 +
            ["toxicophore"] * 2276 +
            ["genomic"] * 3984
        )
        feature_origin = np.array(feature_origin)

        #For Sample Names
        index_names = labels.index.tolist()

Die Labels fügen wir manuell hinzu und führen dazu die Klasse feature origin ein, um uns zu merken zu welchem Feature das ganze VOR dem Preprocessing gehört hat.

In [ ]:
        print(f"Creating folds using: \nnorm: {norm_method}\ntest_fold: {self.test_fold}\nval_fold: {self.val_fold}\n")
        # Get indices for different splits
        idx_tr = np.where(np.logical_and(labels['fold'] != self.test_fold,
                         labels['fold'] != self.val_fold))[0]
        idx_val = np.where(labels['fold'] == self.val_fold)[0]
        idx_train = np.where(labels['fold'] != self.test_fold)[0]
        idx_test = np.where(labels['fold'] == self.test_fold)[0]

In [ ]:
        # Split features
        X_tr = X[idx_tr]
        X_val = X[idx_val]
        X_train = X[idx_train]
        X_test = X[idx_test]

Wir ordnen jetzt die Indixe zu tatsächlichen Daten zu

In [ ]:
        # Split labels
        y_tr = labels.iloc[idx_tr]['synergy'].values
        y_val = labels.iloc[idx_val]['synergy'].values
        y_train = labels.iloc[idx_train]['synergy'].values
        y_test = labels.iloc[idx_test]['synergy'].values

Und ziehen uns die zugehörigen Zielwerte (Labels) heraus

In [ ]:
        # Normalize data
        if norm_method == "tanh_norm":
            # First normalize X_tr to obtain feat_filt (non-informative feature filter)
            X_tr, mean, std, mean2, std2, feat_filt = self.normalize(X_tr, norm=norm_method)
            # Filter feature origin list in sync with the removal of non-informative features
            filtered_feature_origin = feature_origin[feat_filt]

            # Then normalize the other datasets using the same parameters and feat_filt
            X_val, _, _, _, _, _ = self.normalize(X_val, mean, std, mean2, std2, feat_filt=feat_filt, norm=norm_method)
            X_train, _, _, _, _, _ = self.normalize(X_train, mean, std, mean2, std2, feat_filt=feat_filt, norm=norm_method)
            X_test, _, _, _, _, _ = self.normalize(X_test, mean, std, mean2, std2, feat_filt=feat_filt, norm=norm_method)
        else:
            X_tr, mean, std, feat_filt = self.normalize(X_tr, norm=norm_method)
            filtered_feature_origin = feature_origin[feat_filt]

            X_val, _, _, _ = self.normalize(X_val, mean, std, feat_filt=feat_filt, norm=norm_method)
            X_train, _, _, _ = self.normalize(X_train, mean, std, feat_filt=feat_filt, norm=norm_method)
            X_test, _, _, _ = self.normalize(X_test, mean, std, feat_filt=feat_filt, norm=norm_method)

X_val wird hier jeweils mit den Parametern aus dem Trainingsdurchlauf normalisiert, um Data Leakage zu vermeiden. Angenommen wir würden auch aus XVal Daten nehmen, um die normalisierung durchzuführen, würde das Modell bereits die Lösung für die Daten kennen und somit besser erscheinen, als es ist.

Wichtig ist hier also, dass die Daten ERST getrennt und DANN die normalisierung berechnet wird. Ansonsten würden bereits statistische Informationen aus der Lösung ins Modell einfließen. Wichtig ist hierbei, dass wir die Parameter aus dem Trainingsdurchlauf weitergeben und NICHT eine erneute Normalisierung für den Validierungsdatensatz machen.

Hier machen wir den inneren und äußeren Trainingsloop.

In [ ]:
        print("Dumping result..")
        with gzip.open('test%dval%dnorm%s.p.gz' % (self.test_fold, self.val_fold, norm_method), 'wb') as f:
            pickle.dump((X_tr, X_val, X_train, X_test, y_tr, y_val, y_train, y_test, index_names, filtered_feature_origin), f)

Als letztes speichern wir einfach die gesplitteten Daten, um diese exakt so wieder aufrufen zu können.

In [ ]:
# Initialize
preprocessor = Preprocessor(test_fold=0, val_fold=1)
# Load and process data
data = preprocessor.create_splits(norm_method='tanh_norm')

Die eigentliche Funktion starten wir jetzt ganz einfach.